# Replica del metodo di Toma, Piltan e Kim (2021)Riproduzione del framework DAE + CNN descritto in *A Deep Autoencoder-Based ConvolutionNeural Network Framework for Bearing Fault Classification in Induction Motors*,Sensors 21, 8453, applicato ai dati reali del KAt-DataCenter dell'Universita di Paderborn.L'obiettivo e la fedelta: stessi cuscinetti della Tabella 2, stessa architettura delleTabelle 3 e 4, stesso protocollo della Sezione 4. Dove il paper non specifica unparametro la scelta e dichiarata esplicitamente nella cella corrispondente.Il notebook gira su Colab: i dati stanno su Drive, il codice viene da GitHub, irisultati (figure e tabelle) tornano su Drive.

In [ ]:
# unrar serve a scompattare gli archivi, scipy a leggere i .mat
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install requests scipy

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    su_colab = True
except ImportError:
    su_colab = False

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'

# Stesse cartelle del notebook 01: prima accanto al notebook, poi su Drive, poi il repo.
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']

percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break

if percorso_codice is None:
    print('codice non trovato in locale, clono il repo')
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'],
                   check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'

sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='02_replica_paper')
dev = f.dispositivo()

print('codice da', percorso_codice)
print('risultati in', P['risultati'])
print('dispositivo', dev)

## 1. Cuscinetti e condizione operativaI 17 cuscinetti sono quelli della Tabella 2 del paper: sei sani, cinque con guastosulla pista esterna, sei sulla pista interna. Sono tutti a danno reale, prodotto daprove di vita accelerate, non a danno artificiale.Il paper dice di avere lavorato su "tre diverse condizioni" ma non indica quali. Ilnumero di segmenti che dichiara, 1320, permette di restringere il campo: il datasetmette a disposizione quattro regimi, e in ciascuno i 17 cuscinetti hanno 20registrazioni da 4 s, cioe 1360 segmenti da un secondo. Con due o piu regimi siarriverebbe ad almeno 2720. Quindi 1320 e compatibile soltanto con una singolacondizione operativa, ed e ragionevole leggere le "tre condizioni" come le tre*classi* di cuscinetto. Qui si usa `N15_M07_F10` (1500 rpm, 0,7 Nm, 1000 N), lacondizione nominale del banco.

In [ ]:
cuscinetti = config.CUSCINETTI_PAPER
regime = config.REGIME_PRINCIPALE

print(len(cuscinetti), 'cuscinetti,', regime, config.REGIMI[regime])
for classe, elenco in config.CUSCINETTI_PER_CLASSE.items():
    print(' ', config.NOMI_CLASSI[classe], '->', len(elenco), ':', ' '.join(elenco))

## 2. Scarico ed estrazione degli archiviGli archivi `.rar` (uno per cuscinetto, 150-180 MB ciascuno) restano su Drive: siscaricano una volta sola e vengono estratti sul disco locale della macchina Colab,che e piu veloce di Drive in lettura e viene comunque azzerato a fine sessione.

In [ ]:
archivi, dimensione = f.scarica_archivi(cuscinetti, P['raw'])
print(len(archivi), 'archivi su Drive |', round(dimensione / 1e9, 2), 'GB')

f.estrai_misure(cuscinetti, P['raw'], P['estratti'])
print(len(f.elenco_registrazioni(cuscinetti, P['estratti'])), 'file .mat estratti')

## 3. Inventario, prima di qualunque calcoloUn'estrazione parziale non genera errori: genera silenziosamente meno dati. Prima dicostruire i segmenti conviene quindi censire quello che c'e davvero, su tutti equattro i regimi, e confrontarlo con quello che ci si aspetta.Il conteggio atteso e 17 cuscinetti x 20 registrazioni x 4 regimi = 1360.

In [ ]:
# Il conteggio si fa sui nomi dei file: non serve aprirli, e su 1360 file la
# differenza fra guardare i nomi e leggere i dati e di qualche minuto.
catalogo = pd.DataFrame([f.metadati_nome(p) for p in
                         f.elenco_registrazioni(cuscinetti, P['estratti'])])
catalogo['classe'] = catalogo['cuscinetto'].map(config.CLASSE_DI)

riepilogo = (catalogo.groupby(['regime', 'classe'])
             .agg(registrazioni=('registrazione', 'count'),
                  cuscinetti=('cuscinetto', 'nunique'))
             .reset_index())
riepilogo['nome_classe'] = riepilogo['classe'].map(dict(enumerate(config.NOMI_CLASSI)))
print(riepilogo.to_string(index=False))
print()

atteso = len(cuscinetti) * 20
print('attese', atteso, 'registrazioni per regime')
for nome_regime, gruppo in catalogo.groupby('regime'):
    print(' ', nome_regime, len(gruppo), 'ok' if len(gruppo) == atteso else 'INCOMPLETO')

per_cuscinetto = catalogo.groupby(['cuscinetto', 'regime']).size().unstack()
incompleti = per_cuscinetto[(per_cuscinetto != 20).any(axis=1)]
print()
if len(incompleti):
    print('cuscinetti con un numero di registrazioni diverso da 20:')
    print(incompleti.to_string())
else:
    print('tutti i', len(per_cuscinetto), 'cuscinetti hanno 20 registrazioni per regime,',
          'totale', len(catalogo))

## 4. Segmenti da un secondoOgni registrazione da 4 s produce quattro segmenti non sovrapposti da 64 000 campioni.Le registrazioni leggermente piu corte di 256 000 campioni danno un segmento in meno:sono poche e vengono contate, non scartate a monte.

In [ ]:
inv = f.inventario(cuscinetti, P['estratti'], regimi=[regime]).reset_index(drop=True)
segmenti, anagrafica = f.costruisci_segmenti(inv, segmenti_per_registrazione=4)

print(len(segmenti), 'segmenti da', segmenti.shape[1], 'campioni')
print('memoria:', round(segmenti.nbytes / 1e6), 'MB')
print()

conteggi = anagrafica.groupby('classe').agg(segmenti=('segmento', 'size'),
                                            registrazioni=('registrazione', 'nunique'),
                                            cuscinetti=('cuscinetto', 'nunique'))
conteggi.index = [config.NOMI_CLASSI[c] for c in conteggi.index]
print(conteggi.to_string())
print()

corte = inv[inv['campioni'] < 4 * config.FS_ATTESO]
print('registrazioni con meno di 256000 campioni:', len(corte),
      '-> segmenti persi:', 4 * len(inv) - len(segmenti))
print('il paper ne dichiara', config.PAPER_SEGMENTI_DICHIARATI,
      '| differenza:', len(segmenti) - config.PAPER_SEGMENTI_DICHIARATI)

## 5. Il segnale, prima di darlo in pasto alla reteDue grandezze servono piu avanti: il valore efficace, perche il residuo dell'arco SELUgli e legato, e il picco, perche determina quanti campioni finiscono sotto il limiteinferiore della SELU.

In [ ]:
descrittive = pd.DataFrame([f.caratteristiche(x) for x in segmenti])
descrittive['classe'] = anagrafica['classe'].values
descrittive['cuscinetto'] = anagrafica['cuscinetto'].values
descrittive['sotto_pavimento'] = np.mean(segmenti < config.PAVIMENTO_SELU, axis=1)

per_classe = descrittive.groupby('classe')[['rms', 'picco', 'crest', 'f_dominante', 'sotto_pavimento']].mean()
per_classe.index = [config.NOMI_CLASSI[c] for c in per_classe.index]
print('media per classe:')
print(per_classe.to_string())
print()
print('media per cuscinetto:')
print(descrittive.groupby('cuscinetto')[['rms', 'picco', 'crest', 'sotto_pavimento']].mean().to_string())
print()
print('valori non finiti:', int(descrittive['non_finiti'].sum()))

## 6. Frame e limite inferiore della SELULa segmentazione fine segue le equazioni 7-9 del paper: un frame contiene un giromeccanico completo. A 1500 rpm e 64 kHz sono 2560 campioni, quindi 25 frame persegmento.La SELU e limitata inferiormente da $-\lambda\alpha \approx -1{,}7581$: il decoder nonpuo produrre in uscita valori piu bassi, qualunque cosa impari. Con corrente nonnormalizzata e picchi intorno a 3 A, una parte dei campioni cade sotto quella sogliaed e irriproducibile per costruzione. Qui si misura quanti sono.Il paper non dice se il segnale venga normalizzato prima di entrare nel DAE. Nonessendoci alcun riferimento a una normalizzazione, qui si usa il segnale come e.

In [ ]:
lunghezza_frame = f.lunghezza_frame(config.REGIMI[regime]['rpm'])
frame_per_segmento = segmenti.shape[1] // lunghezza_frame
frame = segmenti.reshape(-1, lunghezza_frame)
classe_del_frame = np.repeat(anagrafica['classe'].values, frame_per_segmento)
segmento_del_frame = np.repeat(np.arange(len(segmenti)), frame_per_segmento)

print('un giro a', config.REGIMI[regime]['rpm'], 'rpm ->', lunghezza_frame, 'campioni')
print(len(frame), 'frame,', frame_per_segmento, 'per segmento')

# il reshape deve conservare l'ordine: il frame j del segmento i sta nella riga i*25+j
prova = np.array_equal(frame[3 * frame_per_segmento + 7],
                       segmenti[3, 7 * lunghezza_frame:8 * lunghezza_frame])
print('il reshape conserva l ordine dei frame:', prova)
print('il reshape inverso restituisce i segmenti:',
      np.array_equal(frame.reshape(len(segmenti), -1), segmenti))
print()

sotto = frame < config.PAVIMENTO_SELU
print('pavimento della SELU:', round(config.PAVIMENTO_SELU, 4))
print('campioni sotto il pavimento:', round(100 * float(np.mean(sotto)), 2), '%')
print('frame che ne contengono almeno uno:',
      round(100 * float(np.mean(sotto.any(axis=1))), 2), '%')

## 7. Le due architettureLe Tabelle 3 e 4 del paper riportano il numero di parametri di ogni strato. Sonosufficienti a ricostruire l'architettura senza ambiguita, e il conteggio totale serveda verifica: se coincide, la ricostruzione e corretta.Due dettagli non sono dichiarati dal paper e vanno ricavati o scelti. Il nucleo delleconvoluzioni si ricava: con un canale in ingresso e 64 filtri con bias, solo un nucleoda 3 da i 256 parametri del primo strato e i 6176 del secondo. L'inizializzazione deipesi va scelta, e qui e LeCun normale, l'unica per cui vale l'auto-normalizzazionedella SELU dimostrata da Klambauer et al.

In [ ]:
dae_prova = f.crea_dae()
cnn_prova = f.crea_cnn(segmenti.shape[1])

print('DAE:', f.conta_parametri(dae_prova), 'parametri')
print('CNN:', f.conta_parametri(cnn_prova), 'parametri')
print()

for nucleo in [2, 3, 4, 5]:
    primo = 64 * (nucleo * 1 + 1)
    secondo = 32 * (nucleo * 64 + 1)
    nota = '  <- coincide con la Tabella 4' if (primo, secondo) == (256, 6176) else ''
    print('nucleo', nucleo, '-> primo strato', primo, 'parametri, secondo', secondo, nota)

del dae_prova, cnn_prova

## 8. I frame sani per addestrare il DAEIl paper addestra il DAE su 2560 frame di cuscinetto sano, divisi 80:20 in 2048 perl'addestramento e 512 per il calcolo dell'errore di validazione. Non dice come venganoscelti fra i 12 000 disponibili, ne se la divisione sia casuale.Qui i 2560 sono estratti a sorte fra tutti i frame sani e poi **mescolati** prima delladivisione. Il mescolamento non e un dettaglio: senza di esso i primi 2048 indicicadrebbero tutti nei primi cuscinetti e la validazione finirebbe per essere compostaquasi interamente da K006, con un valore efficace sistematicamente piu basso dellamedia. L'errore di validazione risulterebbe piu basso di quello di addestramento, esarebbe un artefatto della divisione, non una proprieta del modello.

In [ ]:
frame_dae = 2560
frame_dae_addestramento = 2048
seme = 0

indici_sani = np.flatnonzero(classe_del_frame == 0)
rng = np.random.default_rng(seme)
scelti = rng.choice(indici_sani, size=frame_dae, replace=False)
rng.shuffle(scelti)

indici_dae_train = scelti[:frame_dae_addestramento]
indici_dae_val = scelti[frame_dae_addestramento:]

visto_dal_dae = np.zeros(len(frame), dtype=bool)
visto_dal_dae[scelti] = True

print('frame sani disponibili:', len(indici_sani),
      '| usati per il DAE:', frame_dae,
      '->', round(100 * frame_dae / len(indici_sani), 1), '%')
print('addestramento', len(indici_dae_train), '| validazione', len(indici_dae_val))
print()

cuscinetto_del_frame = np.repeat(anagrafica['cuscinetto'].values, frame_per_segmento)
composizione = pd.DataFrame({
    'addestramento': pd.Series(cuscinetto_del_frame[indici_dae_train]).value_counts(),
    'validazione': pd.Series(cuscinetto_del_frame[indici_dae_val]).value_counts(),
}).fillna(0).astype(int).sort_index()
print('da quali cuscinetti provengono i frame:')
print(composizione.to_string())

## 9. Addestramento dei due autoencoderIl paper prescrive l'attivazione SELU su tutti gli strati, uscita compresa (Tabella 3).Per stabilire quanta parte del residuo dipenda dal limite inferiore della SELU e quantadalla modellazione del sistema si addestra un secondo autoencoder identico in tuttotranne l'ultima attivazione, che e lineare. Stessi dati, stesso seme, stesse epoche:l'unica differenza e quella.500 epoche fisse, senza arresto anticipato, come dichiarato. Il passo di apprendimento(0,0003) e la dimensione del lotto (256) non sono dichiarati per il DAE: il lotto da 64che il paper indica si riferisce esplicitamente alla CNN.

In [ ]:
frame_train = frame[indici_dae_train]
frame_val = frame[indici_dae_val]

dae = {}
curve = {}
for nome, uscita_selu in [('selu', True), ('lineare', False)]:
    print('DAE con uscita', nome)
    modello, c_train, c_val = f.addestra_dae(frame_train, frame_val,
                                             uscita_selu=uscita_selu,
                                             epoche=500, lotto=256, passo=3e-4,
                                             seme=seme, dev=dev, stampa_ogni=50)
    dae[nome] = modello
    curve[nome] = {'addestramento': c_train, 'validazione': c_val}
    print()

## 10. Il residuoIl residuo e la differenza campione per campione fra il segnale e la sua ricostruzione(equazione 15). Il paper riporta un residuo medio di 0,104 per lo stato normale, 0,386per la pista esterna e 0,479 per la interna: rapporti di 3,71 e 4,61 rispetto al sano,ed e questa separazione a rendere il problema facile per la CNN a valle.La Sezione 3.4 del paper precisa che le istanze usate per generare il residuo sonodiverse da quelle di addestramento. Qui il residuo viene calcolato su tutti i frame,perche i segmenti servono interi alla CNN, ma il residuo medio della classe normaleviene riportato in entrambi i modi: su tutti i frame sani e sui soli frame che il DAEnon ha mai visto. La differenza fra i due valori misura l'entita esatta di questoscostamento, invece di lasciarla all'argomentazione.

In [ ]:
residui = {}
mse_frame = {}
for nome in dae:
    residui[nome] = f.calcola_residui(dae[nome], frame, dev=dev)
    mse_frame[nome] = f.mse_per_frame(residui[nome])

righe = []
for nome in dae:
    for classe, etichetta in enumerate(config.NOMI_CLASSI):
        del_classe = classe_del_frame == classe
        righe.append({
            'uscita': nome,
            'classe': etichetta,
            'residuo_tutti': float(np.mean(mse_frame[nome][del_classe])),
            'residuo_mai_visti': float(np.mean(mse_frame[nome][del_classe & ~visto_dal_dae])),
            'paper': config.PAPER_RESIDUO[etichetta],
        })

tabella_residui = pd.DataFrame(righe)
for nome in dae:
    parte = tabella_residui['uscita'] == nome
    base_tutti = tabella_residui.loc[parte, 'residuo_tutti'].iloc[0]
    base_visti = tabella_residui.loc[parte, 'residuo_mai_visti'].iloc[0]
    tabella_residui.loc[parte, 'rapporto'] = tabella_residui.loc[parte, 'residuo_tutti'] / base_tutti
    tabella_residui.loc[parte, 'rapporto_mai_visti'] = tabella_residui.loc[parte, 'residuo_mai_visti'] / base_visti
tabella_residui['rapporto_paper'] = tabella_residui['paper'] / config.PAPER_RESIDUO['normale']

print(tabella_residui.to_string(index=False))
print()

scarto = 100 * abs(tabella_residui['residuo_mai_visti'] - tabella_residui['residuo_tutti']) / tabella_residui['residuo_tutti']
print('scarto massimo fra i due modi di calcolare il residuo:', round(float(scarto.max()), 2), '%')

### Da dove viene il residuo dell'arco SELUDue misure dicono se il residuo porti informazione sul cuscinetto oppure sull'ampiezzadel segnale: la quota di energia del residuo che si concentra sui campioni sotto ilpavimento della SELU, e la correlazione fra il valore efficace di un segmento e il suoresiduo. L'arco lineare fa da termine di paragone.

In [ ]:
for nome in dae:
    r = residui[nome].astype(np.float64) ** 2
    quota = 100 * float(np.sum(r[sotto]) / np.sum(r))
    mse_segmento = mse_frame[nome].reshape(len(segmenti), frame_per_segmento).mean(axis=1)
    correlazione = float(np.corrcoef(descrittive['rms'].values, mse_segmento)[0, 1])
    print('uscita', nome)
    print('   energia del residuo che sta sui campioni sotto il pavimento:',
          round(quota, 2), '%')
    print('   correlazione fra valore efficace del segmento e suo residuo:',
          round(correlazione, 3))

### La stessa rappresentazione della Figura 8 del paperLa Figura 8 dell'articolo mostra, per un esemplare di ciascuna classe, il segnalegrezzo, la ricostruzione del DAE e il residuo. La ricostruzione si ottiene dai datigia calcolati, perche il residuo e definito come differenza: x_ricostruito = x - r.E la rappresentazione piu diretta dell'ipotesi sul pavimento: se e giusta, laricostruzione deve apparire tagliata di netto in basso, mentre il segnale prosegue.

In [ ]:
esempi = [('K001', 'normale'), ('KA04', 'pista esterna'), ('KI04', 'pista interna')]
punti = 1000

fig, assi = plt.subplots(2, 2, figsize=(11, 6.5))
for ax, (cuscinetto, descrizione) in zip(assi.ravel()[:3], esempi):
    riga = int(anagrafica.index[anagrafica['cuscinetto'] == cuscinetto][0])
    primo = riga * frame_per_segmento
    x = frame[primo][:punti]
    r = residui['selu'][primo][:punti]
    ax.plot(x, lw=0.8, color=f.COLORI['scuro'], label='segnale')
    ax.plot(x - r, lw=0.8, color=f.COLORI['nostro'], label='ricostruito dal DAE')
    ax.plot(r, lw=0.8, color=f.COLORI['interno'], label='residuo')
    ax.axhline(config.PAVIMENTO_SELU, color=f.COLORI['accento'], ls='--', lw=0.9)
    ax.set_title(descrizione + '  (' + cuscinetto + ')', fontsize=9.5, loc='left')
    ax.set_xlabel('campione'); ax.set_ylabel('corrente (A)')
assi.ravel()[0].legend(fontsize=7, loc='lower right')

ax = assi.ravel()[3]
inizio = 0
for classe, nome in enumerate(config.NOMI_CLASSI):
    riga = int(anagrafica.index[anagrafica['classe'] == classe][0])
    r = residui['selu'][riga * frame_per_segmento][:100]
    ax.plot(np.arange(inizio, inizio + 100), r, lw=0.8,
            color=f.COLORI[nome], label=nome)
    inizio = inizio + 100
ax.set_title('Residuo su 100 campioni per classe', fontsize=9.5, loc='left')
ax.set_xlabel('campione'); ax.set_ylabel('residuo (A)')
ax.legend(fontsize=7)

f.salva_figura(fig, 'segnale_ricostruzione_residuo', P['figure'])
plt.show()

## 11. Suddivisione e classificazioneIl paper suddivide 80/20 i campioni di residuo, senza vincoli sulla provenienza. Poicheogni registrazione da 4 s produce quattro segmenti consecutivi, segmenti della stessaregistrazione finiscono sia in addestramento sia in verifica. E una fuga di informazione,e viene riprodotta deliberatamente: correggerla qui renderebbe impossibile capire se loscarto rispetto al paper dipenda da questa scelta o da altro. Il confronto consuddivisioni piu severe e nel notebook dell'indagine.La stessa suddivisione viene usata da tutti i metodi confrontati, cosi il confrontoresta controllato.Una differenza da dichiarare: qui la suddivisione e stratificata per classe, cioe laproporzione fra le tre classi e la stessa in addestramento e in verifica. Il paper dicesoltanto "80% of these residual signal samples", senza precisare. Con un sorteggio nonstratificato la composizione dell'insieme di verifica varia da un seme all'altro e leaccuratezze diventano meno confrontabili fra loro; la stratificazione toglie questafonte di rumore senza toccare la fuga di informazione, che e il punto in discussione.

In [ ]:
etichette = anagrafica['classe'].values.astype(np.int64)
indici_train, indici_test = f.suddividi(anagrafica, livello='segmento',
                                        frazione_test=0.2, seme=seme)

print(f.sovrapposizione(anagrafica, indici_train, indici_test))
print()
print('composizione della verifica:')
print(pd.Series(etichette[indici_test]).map(dict(enumerate(config.NOMI_CLASSI)))
      .value_counts().to_string())
print()
print('accuratezza della classe piu numerosa:',
      round(100 * f.baseline_degenere(etichette[indici_test])['accuratezza'], 2), '%')

In [ ]:
residui_segmento = {nome: residui[nome].reshape(len(segmenti), -1) for nome in residui}

ingressi = [('DAE + residuo + CNN', residui_segmento['selu']),
            ('DAE + residuo + CNN, uscita lineare', residui_segmento['lineare']),
            ('segnale grezzo + CNN', segmenti)]

risultati = {}
confusioni = {}
for etichetta, dati in ingressi:
    _, previste, vere, _ = f.addestra_cnn(dati, etichette, indici_train, indici_test,
                                          epoche=500, lotto=64, passo=3e-4,
                                          seme=seme, dev=dev, stampa_ogni=100,
                                          etichetta=etichetta)
    m = f.metriche(vere, previste)
    risultati[etichetta] = {chiave: m[chiave] for chiave in
                            ['richiamo', 'precisione', 'f1', 'accuratezza']}
    confusioni[etichetta] = m['confusione']
    print(m['report'])
    print(pd.DataFrame(m['confusione'], index=config.NOMI_CLASSI,
                       columns=config.NOMI_CLASSI).to_string())
    print()

## 12. Ablazione con le caratteristiche statisticheLa Tabella 5 del paper elenca dieci grandezze estratte dal residuo, date poi in pasto auna macchina a vettori di supporto, a una foresta casuale e a un k-nearest neighbor.Sono tutte e dieci: valore efficace, energia, deviazione standard, curtosi, varianza,asimmetria, fattore di cresta, entropia di Shannon, fattore di forma ed entropialog-energetica.Vale la pena guardare quanto siano davvero indipendenti fra loro, perche quattro di essesono funzioni monotone della stessa quantita.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

caratteristiche = f.feature_tabella5(residui_segmento['selu'])
correlazioni = pd.DataFrame(np.corrcoef(caratteristiche, rowvar=False),
                            index=f.NOMI_FEATURE_TABELLA5,
                            columns=f.NOMI_FEATURE_TABELLA5)
print('correlazione fra le dieci caratteristiche (valore assoluto sopra 0,99):')
alta = correlazioni.abs().where(np.triu(np.ones(correlazioni.shape), k=1).astype(bool)).stack()
print(alta[alta > 0.99].round(4).to_string())
print()

for nome in residui_segmento:
    caratteristiche = f.feature_tabella5(residui_segmento[nome])
    scalatore = StandardScaler().fit(caratteristiche[indici_train])
    X_train = scalatore.transform(caratteristiche[indici_train])
    X_test = scalatore.transform(caratteristiche[indici_test])

    classificatori = {'SVM': SVC(random_state=seme),
                      'RF': RandomForestClassifier(n_estimators=200, random_state=seme),
                      'KNN': KNeighborsClassifier()}
    print('caratteristiche del residuo, uscita', nome)
    for sigla, classificatore in classificatori.items():
        classificatore.fit(X_train, etichette[indici_train])
        m = f.metriche(etichette[indici_test], classificatore.predict(X_test))
        print('   residuo + caratteristiche +', sigla,
              '-> accuratezza', round(100 * m['accuratezza'], 2), '%',
              '| richiamo', round(m['richiamo'], 2),
              '| precisione', round(m['precisione'], 2),
              '| F1', round(m['f1'], 2))
        if nome == 'selu':
            risultati['residuo + feature + ' + sigla] = {
                chiave: m[chiave] for chiave in ['richiamo', 'precisione', 'f1', 'accuratezza']}
            confusioni['residuo + feature + ' + sigla] = m['confusione']
    print()

## 12b. Ripetibilita, come nella Figura 9a del paperLa Figura 9a dell'articolo riporta la distribuzione dell'accuratezza su 100esperimenti. E un'informazione preziosa, perche mostra che i tre metodi basati sullecaratteristiche statistiche hanno una dispersione molto ampia, mentre il metodoproposto resta compatto attorno al 99%.Qui la stessa cosa si puo fare a costo quasi nullo per i tre classificatori, cambiando100 volte la suddivisione: bastano un paio di minuti. Per la CNN non e praticabile,perche un solo addestramento richiede una ventina di minuti, quindi il suo valoreresta quello di una singola esecuzione e va letto con questa avvertenza.

In [ ]:
ripetizioni = 100
caratteristiche = f.feature_tabella5(residui_segmento['selu'])
dispersione = {'SVM': [], 'RF': [], 'KNN': []}

for ripetizione in range(ripetizioni):
    tr, te = f.suddividi(anagrafica, livello='segmento', frazione_test=0.2,
                         seme=ripetizione)
    scalatore = StandardScaler().fit(caratteristiche[tr])
    modelli = {'SVM': SVC(random_state=seme),
               'RF': RandomForestClassifier(n_estimators=200, random_state=seme),
               'KNN': KNeighborsClassifier()}
    for sigla, modello in modelli.items():
        modello.fit(scalatore.transform(caratteristiche[tr]), etichette[tr])
        dispersione[sigla].append(
            100 * float(np.mean(modello.predict(scalatore.transform(caratteristiche[te]))
                                == etichette[te])))

sintesi = pd.DataFrame({
    'metodo': list(dispersione),
    'mediana': [float(np.median(v)) for v in dispersione.values()],
    'minimo': [float(np.min(v)) for v in dispersione.values()],
    'massimo': [float(np.max(v)) for v in dispersione.values()],
    'scarto_tipo': [float(np.std(v)) for v in dispersione.values()],
    'paper': [config.PAPER_TABELLA6['residuo + feature + ' + m]['accuratezza']
              for m in dispersione],
})
print('accuratezza su', ripetizioni, 'suddivisioni diverse:')
print(sintesi.round(2).to_string(index=False))

### Le due figure di confronto con la Figura 9 del paper

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(11, 3.9))

dati_box = [dispersione[m] for m in dispersione]
disegno = assi[0].boxplot(dati_box, patch_artist=True, widths=0.55)
for corpo in disegno['boxes']:
    corpo.set_facecolor(f.COLORI['nostro']); corpo.set_alpha(0.75)
    corpo.set_edgecolor(f.COLORI['scuro'])
for chiave in ['medians', 'whiskers', 'caps']:
    for linea in disegno[chiave]:
        linea.set_color(f.COLORI['scuro'])
assi[0].plot(np.arange(1, len(dispersione) + 1), sintesi['paper'], 'o',
             color=f.COLORI['accento'], label='valore dichiarato dal paper')
assi[0].set_xticks(np.arange(1, len(dispersione) + 1))
assi[0].set_xticklabels(list(dispersione))
assi[0].set_ylabel('accuratezza [%]')
assi[0].set_title('Dispersione su ' + str(ripetizioni) + ' suddivisioni', fontsize=10)
assi[0].legend(fontsize=7)

nostra = confusioni['DAE + residuo + CNN']
paper = np.array(config.PAPER_CONFUSIONE)
for ax, (titolo, matrice) in zip([assi[1]], [('replica', nostra)]):
    quote = matrice / matrice.sum(axis=1, keepdims=True)
    ax.imshow(quote, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, int(matrice[i, j]), ha='center', va='center', fontsize=9,
                    color='white' if quote[i, j] > 0.5 else f.COLORI['scuro'])
    ax.set_xticks(range(3)); ax.set_xticklabels(config.NOMI_CLASSI, fontsize=8)
    ax.set_yticks(range(3)); ax.set_yticklabels(config.NOMI_CLASSI, fontsize=8)
    ax.set_xlabel('classe prevista'); ax.set_ylabel('classe vera')
    ax.set_title('Matrice di confusione, ' + titolo, fontsize=10)

f.salva_figura(fig, 'dispersione_e_confusione', P['figure'])
plt.show()

print('per confronto, la matrice della Figura 9b del paper (238 segmenti, un errore):')
print(pd.DataFrame(paper, index=config.NOMI_CLASSI, columns=config.NOMI_CLASSI).to_string())

## 13. Riepilogo, figure e salvataggio

In [ ]:
righe = []
for metodo, valori in risultati.items():
    riga = {'metodo': metodo}
    for chiave in ['richiamo', 'precisione', 'f1', 'accuratezza']:
        riga['nostro_' + chiave] = valori[chiave] * (100 if chiave == 'accuratezza' else 1)
        atteso = config.PAPER_TABELLA6.get(metodo)
        riga['paper_' + chiave] = atteso[chiave] if atteso else np.nan
    righe.append(riga)

confronto = pd.DataFrame(righe)
confronto['scarto_accuratezza'] = confronto['nostro_accuratezza'] - confronto['paper_accuratezza']

colonne = ['metodo', 'nostro_richiamo', 'paper_richiamo', 'nostro_precisione',
           'paper_precisione', 'nostro_f1', 'paper_f1',
           'nostro_accuratezza', 'paper_accuratezza', 'scarto_accuratezza']
print(confronto[colonne].round(3).to_string(index=False))

In [ ]:
# Dove finisce l'errore di ricostruzione? Se l'ipotesi del pavimento e giusta, il
# residuo dell'arco SELU deve crescere bruscamente sotto -1,7581 e quello dell'arco
# lineare no.
fig, assi = plt.subplots(1, 2, figsize=(11, 3.8))

for nome, colore in [('selu', f.COLORI['nostro']), ('lineare', f.COLORI['normale'])]:
    assi[0].plot(curve[nome]['addestramento'], lw=1.1, color=colore,
                 label='addestramento, uscita ' + nome)
    assi[0].plot(curve[nome]['validazione'], lw=1.1, ls='--', color=colore,
                 label='validazione, uscita ' + nome)
assi[0].set_yscale('log')
assi[0].set_xlabel('epoca'); assi[0].set_ylabel('errore quadratico medio')
assi[0].set_title('Addestramento dei due autoencoder')
assi[0].legend(fontsize=7)

campionati = np.random.default_rng(0).choice(len(frame), 4000, replace=False)
valori = frame[campionati].ravel()
bordi = np.linspace(np.percentile(valori, 0.1), np.percentile(valori, 99.9), 60)
centri = (bordi[:-1] + bordi[1:]) / 2
posizione = np.digitize(valori, bordi) - 1
dentro = (posizione >= 0) & (posizione < len(centri))
quanti = np.bincount(posizione[dentro], minlength=len(centri))

for nome, colore in [('selu', f.COLORI['nostro']), ('lineare', f.COLORI['normale'])]:
    quadrati = (residui[nome][campionati].ravel().astype(np.float64)) ** 2
    somma = np.bincount(posizione[dentro], weights=quadrati[dentro], minlength=len(centri))
    assi[1].plot(centri, somma / np.maximum(quanti, 1), label='uscita ' + nome, color=colore)
assi[1].axvline(config.PAVIMENTO_SELU, color=f.COLORI['accento'], ls='--', lw=1,
                label='pavimento della SELU')
assi[1].set_yscale('log')
assi[1].set_xlabel('valore della corrente (A)')
assi[1].set_ylabel('residuo quadratico medio')
assi[1].set_title('Da dove viene il residuo')
assi[1].legend(fontsize=7)

f.salva_figura(fig, 'residuo_e_pavimento', P['figure'])
plt.show()

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(11, 3.8))

larghezza = 0.38
posizioni = np.arange(3)
for spostamento, (nome, colore) in zip([-larghezza / 2, larghezza / 2],
                                       [('selu', f.COLORI['nostro']),
                                        ('lineare', f.COLORI['normale'])]):
    parte = tabella_residui[tabella_residui['uscita'] == nome]
    assi[0].bar(posizioni + spostamento, parte['rapporto'], larghezza,
                label='uscita ' + nome, color=colore)
assi[0].plot(posizioni, tabella_residui[tabella_residui['uscita'] == 'selu']['rapporto_paper'],
             'o--', color=f.COLORI['accento'], label='paper')
assi[0].set_xticks(posizioni); assi[0].set_xticklabels(config.NOMI_CLASSI)
assi[0].set_ylabel('residuo rapportato al sano')
assi[0].set_title('Separazione delle classi nel residuo')
assi[0].legend(fontsize=7)

ordine = confronto.sort_values('nostro_accuratezza')
posizioni = np.arange(len(ordine))
assi[1].barh(posizioni - 0.2, ordine['nostro_accuratezza'], 0.4, label='replica', color=f.COLORI['nostro'])
assi[1].barh(posizioni + 0.2, ordine['paper_accuratezza'], 0.4, label='paper', color=f.COLORI['normale'])
assi[1].set_yticks(posizioni)
assi[1].set_yticklabels([m.replace(', uscita', '\n uscita') for m in ordine['metodo']], fontsize=7)
assi[1].set_xlabel('accuratezza [%]')
assi[1].set_title('Confronto con i valori dichiarati')
assi[1].legend(fontsize=7)

f.salva_figura(fig, 'replica_riepilogo', P['figure'])
plt.show()

In [ ]:
f.salva_tabella(confronto, 'confronto_accuratezza', P['tabelle'])
f.salva_tabella(tabella_residui, 'confronto_residui', P['tabelle'])
f.salva_tabella(riepilogo, 'inventario_regimi', P['tabelle'])
f.salva_tabella(sintesi, 'dispersione_ripetizioni', P['tabelle'])
f.salva_tabella(anagrafica, 'anagrafica_segmenti', P['tabelle'])
f.salva_tabella(descrittive.groupby('cuscinetto')[['rms', 'picco', 'crest', 'sotto_pavimento']].mean().reset_index(),
                'statistiche_per_cuscinetto', P['tabelle'])

import torch
for nome in dae:
    percorso = os.path.join(P['risultati'], 'dae_uscita_' + nome + '.pt')
    torch.save(dae[nome].state_dict(), percorso)
    print('modello salvato:', percorso)

print()
for cartella in [P['figure'], P['tabelle'], P['risultati']]:
    for nome in sorted(os.listdir(cartella)):
        percorso = os.path.join(cartella, nome)
        if os.path.isfile(percorso):
            print(nome, round(os.path.getsize(percorso) / 1e6, 2), 'MB')